# IWI experiment · Evaluation 1: all models, pretrained against pretrained and scratch against scratch

This is the first of three notebooks that evaluate the models of 04 to 15 on their test predictions. It finds every finished run, collects its test predictions (every cluster predicted by the model of the fold in which it is a test cluster) into one table, and compares the models on the whole continent: **the runs from pretrained weights with each other, and the runs from random weights (scratch) with each other**. 17 compares them per country and 18 for rural and urban clusters, both from the table that this notebook writes.

| family | notebooks | models | inputs | baseline |
|---|---|---|---|---|
| `ten_models` | 11–15 | ResNet-18/34/50, Swin v2-B, U-Net, ViT-S/16, ResNet-18 + SatCLIP, SatCLIP only, MOSAIKS, OlmoEarth; the baseline models VGG-16-BN and MLP on band statistics | Landsat + NL (12); + 6 wavelet bands (11); + 5 DFT bands (13); + peak-aligned NDVI (14, every alignment and input set found); + temporal transformer (15, every position setting found) | Landsat + NL (12) |
| `table2` | 04, 05 | VGG-16 (04), ResNet-18 finetune and frozen (05) | the rows of Table 2 (VIIRS clusters, 2014 and later) | `base` = Landsat + NL |
| `ridge` | 09, 10 | ridge on the NL features of 06 and the Landsat features of 07 (09) or 08 (10) | the rows of 07 and 08 | `landsat` = NL + Landsat features |
| `single` | 06–08 | ResNet-18 on NL (06); ResNet-18 (07) and ViT-S/16 (08) on Landsat | NL only; Landsat only and its rows | none (shown for reference) |

An **input** (combination) is written as what the model sees: `L+NL` is the baseline, the 7 Landsat bands and the nighttime light; `L+NL+wavelet`, `L+NL+DFT`, `L+NL+peak:pixel11/wavelet6`, `L+NL+TT:cyclic` add the NDVI bands of 11, 13, 14 and 15; `L+NL+NDVI`, `L+NL+HFC_max`, … are the other rows of 04/05 and 09/10.

**Pretrained and scratch.** Every run set takes its start from its `result.json` files (`from_scratch`, or `pretrained` in 06 and 07; runs written before these keys existed started from pretrained weights). The two kinds are never mixed in a comparison: the leaderboards, the gains over the baseline and the comparisons of models are made within the runs from pretrained weights and within the runs from scratch. MOSAIKS and the MLP baseline have no pretrained weights and `satclip_only` reads only the fixed SatCLIP values, so their two runs differ only by training noise. The ridge runs of 09 and 10 have no `result.json`: their start is that of the networks whose features they used, found by matching the network predictions in their test file to the runs of 06–08 (one pretrained and one scratch network: `mixed`).

**What is compared, on the whole continent:**

1. **Leaderboard** of every start (section 4): R² of the pooled test predictions of the five folds, the mean and sd of the R² of the single test folds, RMSE, MAE and bias (predicted − observed) in IWI points.
2. **Every input against the baseline** (section 5), with the same model and start: ΔR² = R²(input) − R²(Landsat + NL).
3. **Every pair of models** with the same input and start (section 6), and the models that cannot be told apart from the best one.
4. **Every model against the two baseline models** (end of section 6; `BASELINE_MODELS`): VGG-16-BN and an MLP on statistics of the input bands, trained with the same inputs, folds and recipe as the ten models (11–15, added 2026-11-16). Each model and input is compared with the baseline model with the **same input** (what the architecture adds) and with the baseline model on **Landsat + NL** (what the architecture and the added bands add together). The other summaries over models (median ΔR²) leave the baseline models out and show them on their own.
5. **Appendix** (section 7): pretrained minus scratch for the same model and input.

**Statistics.** Two runs are compared on the test clusters they share (paired), so ΔR² is their difference in squared error over the same sum of squares. The 95 % interval comes from `BOOTSTRAP` Poisson resamples of 03's spatial groups (clusters closer than 9.5 km form one group, and a group lies in one fold), the same resamples for every comparison. `p` is the two-sided bootstrap p-value, `q` the Benjamini–Hochberg q-value within a table (start and family; the comparisons with a baseline model of `BASELINE_MODELS` have their own, so the q-values of the other models do not change when the baselines are added), and `folds better` counts the test folds in which the run has the higher R². A difference is `better` or `worse` when its interval excludes 0, `n.s.` otherwise. The intervals cover the sampling of the test clusters, not the randomness of training: a model trained twice with another seed would differ a little too, and the ΔR² of `satclip_only` between inputs (it reads no band) shows how much.

**Checks.** A run is used only if its test clusters, folds and IWI agree with `train_clusters_folds.csv` (`DROP_STALE`): runs made with older folds are listed and left out. A run with fewer than five test folds is used with the folds it has; every comparison is made on the clusters both runs have.

**Output,** in `evaluation/<periods>/` next to the run folders (`EVAL_DIR`):

* `eval_runs.csv`: every run set found (family, notebook, model, input, start, test folds, test clusters, problems, used or not, folder) and `eval_test_predictions.csv.gz`: one row per cluster of 03 (country, rural or urban, fold, spatial group, IWI, …) and one column per used run with its test prediction. 17 and 18 read these two files.
* `overall_metrics.csv` (every run, with its gain over the baseline), `overall_pairwise_models.csv`, `overall_best_models.csv`, `overall_vs_baseline_models.csv` (every model against the baseline models) and `overall_vs_baseline_models_summary.csv` (its summary over the ten models), `overall_pretrained_minus_scratch.csv`, and the plots `overall_r2_<family>.png`, `overall_vs_baseline_<start>_<family>.png`, `overall_pairwise_<start>.png`, `overall_vs_baseline_models_<start>.png`, `overall_pretrained_minus_scratch.png`.

**Running.** No GPU: a CPU needs a few minutes (reading the predictions takes most of it). Run it again whenever more runs of 04–15 have finished, then 17 and 18.

# Import libraries

In [ ]:
import os
import re
import glob
import json
import time
import uuid

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap, Normalize
from matplotlib.lines import Line2D
from IPython.display import display, Image

pd.set_option('display.max_rows', 500)
pd.set_option('display.max_columns', 80)
pd.set_option('display.width', 250)
pd.set_option('display.max_colwidth', 60)

# Settings

In [ ]:
# ---- input / output ------------------------------------------------------------------------
WORK_DIR  = '/path/to/data/'
FOLDS_CSV = os.path.join(WORK_DIR, 'train_clusters_folds.csv')   # written by 03_Spatial_Folds
PERIODS   = 'DMSP_VIIRS'   # the run folders of 06-15 that are read, <periods>_<start>...: both NL periods, as in 06-15
EVAL_DIR  = os.path.join(WORK_DIR, 'evaluation', PERIODS)       # 16 writes the collected predictions here, and
                                                                 # 16-18 their tables and plots

# ---- evaluation ----------------------------------------------------------------------------------
TARGET       = 'iwi'           # the label of FOLDS_CSV that every model predicts
FOLDS        = ['A', 'B', 'C', 'D', 'E']
BASELINE     = 'L+NL'          # the baseline input: the 7 Landsat bands and NL (12; the row 'base' of 04 and 05; the
                               # ridge on NL + Landsat features of the row 'landsat' of 09 and 10)
BOOTSTRAP    = 1000            # resamples of 03's spatial groups for the 95 % intervals (0: no intervals)
SEED         = 0               # the resamples: the same for every comparison, so all comparisons are paired
MIN_N        = 30              # an R2 needs at least this many test clusters (a country, its rural or urban clusters)
MIN_FOLD_N   = 10              # a test fold counts in 'folds better' with at least this many of the clusters
NOISE_MODELS = ['satclip_only']   # read no band, only the coordinates: their change between inputs is training noise
                                  # (left out of the summaries over models, shown as 'noise dR2')
BASELINE_MODELS = ['vgg16bn', 'mlp']   # the baseline models of 11-15 (added 2026-11-16: VGG-16-BN and an MLP on band
                                       # statistics, on the same inputs): in every table and plot of the models and
                                       # compared with every model (16, section 6), but left out of the summaries
                                       # over models (shown there on their own: 'vgg16bn dR2', 'mlp dR2'), so these
                                       # stay those of the ten models
SUMMARY_SKIP = NOISE_MODELS + BASELINE_MODELS   # the models that the summaries over models leave out

# ---- what to collect (section 2) ---------------------------------------------------------------------------------------
FAMILIES    = ['ten_models', 'table2', 'ridge', 'single']   # the families of the table at the top
DROP_STALE  = True     # leave out a run whose test clusters, folds or IWI do not agree with FOLDS_CSV (older folds)
RIDGE_START = {}       # ridge_runs/<name> -> 'pretrained', 'scratch' or 'mixed', for a ridge run whose networks section 2
                       # cannot find among the runs of 06-08 (it is 'unknown' otherwise)
CROSS_START = True     # section 7: pretrained minus scratch for the same model and input

os.makedirs(EVAL_DIR, exist_ok=True)
print('runs from', WORK_DIR, '| periods:', PERIODS, '| families:', FAMILIES, '| output ->', EVAL_DIR)
print(f'bootstrap: {BOOTSTRAP} resamples | R2 from {MIN_N} test clusters | baseline input: {BASELINE}')

In [ ]:
# ---- the models and the inputs: names, labels and order ---------------------------------------------------------------
TEN_MODELS = ['resnet18', 'resnet34', 'resnet50', 'swin', 'unet', 'vit', 'satclip', 'satclip_only', 'mosaiks',
              'olmoearth']                                       # the models of 11-15, in their order
MODEL_LABEL = {'resnet18': 'ResNet-18', 'resnet34': 'ResNet-34', 'resnet50': 'ResNet-50', 'swin': 'Swin v2-B',
               'unet': 'U-Net', 'vit': 'ViT-S/16', 'satclip': 'ResNet-18 + SatCLIP', 'satclip_only': 'SatCLIP only',
               'mosaiks': 'MOSAIKS', 'olmoearth': 'OlmoEarth', 'vgg16bn': 'VGG-16-BN', 'mlp': 'MLP (band stats)',
               'vgg16': 'VGG-16 (04)',
               'resnet18_finetune': 'ResNet-18 finetune (05)', 'resnet18_frozen': 'ResNet-18 frozen (05)',
               'nl_resnet18': 'ResNet-18 on NL (06)', 'landsat_resnet18': 'ResNet-18 on Landsat (07)'}
FAMILY_ORDER = ['ten_models', 'table2', 'ridge', 'single']
FAMILY_LABEL = {'ten_models': 'the models of 11-15', 'table2': 'the Table 2 networks of 04 and 05',
                'ridge': 'the ridge on network features of 09 and 10', 'single': 'the NL or Landsat networks of 06-08'}
START_ORDER  = ['pretrained', 'scratch', 'mixed', 'unknown']
MAIN_INPUTS  = ['L+NL+wavelet', 'L+NL+DFT', 'L+NL+peak:pixel11/wavelet6', 'L+NL+TT:cyclic', 'L+NL+NDVI']
SINGLE_BANDS = ['LFC_min', 'LFC_max', 'LFC_mean', 'HFC_min', 'HFC_max', 'HFC_mean']   # the order of the band cache


def model_label(m):
    """Display name of a model key."""
    if m in MODEL_LABEL:
        return MODEL_LABEL[m]
    if m.startswith('ridge:'):
        return f'ridge, {m[6:]}'
    if m.startswith('landsat_vits16_'):
        return f'ViT-S/16 on Landsat (08, {m[15:]})'
    return m


OTHER_MODELS = ['vgg16', 'resnet18_finetune', 'resnet18_frozen', 'nl_resnet18', 'landsat_resnet18']


def model_order(m):
    """Sort key of a model: the ten models of 11-15 in their order and the baseline models, then those of 04-08, then
    the others by name."""
    if m in TEN_MODELS:
        return (0, TEN_MODELS.index(m), '')
    if m in BASELINE_MODELS:
        return (0, len(TEN_MODELS) + BASELINE_MODELS.index(m), '')
    if m in OTHER_MODELS:
        return (1, OTHER_MODELS.index(m), '')
    return (2, 0, m)


def combo_order(c):
    """Sort key of an input: the baseline, what 11, 13, 14 and 15 add to it, the rows of 04/05 and 09/10, then the
    inputs without NL (06-08)."""
    if c == BASELINE:
        return (0, 0, '')
    if not c.startswith(BASELINE + '+'):
        return (10, ['NL', 'L'].index(c), '') if c in ('NL', 'L') else (12, 0, c)
    tail = c[len(BASELINE) + 1:]
    if tail == 'wavelet':
        return (1, 0, '')
    if tail == 'DFT':
        return (2, 0, '')
    if tail.startswith('peak:'):
        align, _, inp = tail[5:].partition('/')
        aligns, inputs = ['pixel11', 'cluster11', 'calendar'], ['wavelet6', 'ndvi23', 'ndvi23_wavelet6']
        return (3, 10 * (inputs.index(inp) if inp in inputs else 3) + (aligns.index(align) if align in aligns else 3),
                tail)
    if tail.startswith('TT:'):
        return (4, {'cyclic': 0, 'none': 1, 'absolute': 2}.get(tail[3:], 3), tail)
    if tail == 'NDVI':
        return (5, 0, '')
    if tail in SINGLE_BANDS:
        return (6, SINGLE_BANDS.index(tail), '')
    return (7, 0, tail)


def short_combo(c):
    """Short name of an input: what it adds to Landsat + NL ('wavelet', 'DFT', 'peak pixel11/wavelet6', 'TT cyclic',
    'NDVI', 'HFC max', ...), and 'Landsat + NL' for the baseline."""
    if c == BASELINE:
        return 'Landsat + NL'
    tail = c[len(BASELINE) + 1:] if c.startswith(BASELINE + '+') else c
    return tail.replace('peak:', 'peak ').replace('TT:', 'TT ').replace('_', ' ')


def combo_label(c):
    """Long name of an input."""
    if c == BASELINE:
        return 'Landsat + NL (baseline)'
    if not c.startswith(BASELINE + '+'):
        return {'NL': 'NL only', 'L': 'Landsat only'}.get(c, c.replace('L+', 'Landsat + ').replace('_', ' '))
    tail = c[len(BASELINE) + 1:]
    if tail.startswith('peak:'):
        return f'Landsat + NL + peak-aligned NDVI, {tail[5:]} (14)'
    if tail.startswith('TT:'):
        return f'Landsat + NL + temporal transformer, {tail[3:]} (15)'
    names = {'wavelet': '6 wavelet bands', 'DFT': '5 DFT bands (13)', 'NDVI': '12 monthly NDVI values'}
    return 'Landsat + NL + ' + names.get(tail, tail.replace('_', ' '))


def sort_runs(frame):
    """Rows of a table with the columns start, family, model and combo in the order of START_ORDER, FAMILY_ORDER,
    model_order and combo_order."""
    keys = [(START_ORDER.index(s) if s in START_ORDER else 9, FAMILY_ORDER.index(f) if f in FAMILY_ORDER else 9,
             model_order(m), combo_order(c))
            for s, f, m, c in zip(frame['start'], frame['family'], frame['model'], frame['combo'])]
    return frame.iloc[sorted(range(len(frame)), key=keys.__getitem__)]

In [ ]:
SCORE_COLS = ['n', f'{TARGET} mean', f'{TARGET} sd', 'R2', 'r2', 'RMSE', 'MAE', 'bias']


def cell_scores(y, p, code, n_cells, min_n=None):
    """Scores of the prediction column p in every cell (code: the cell of every cluster, -1 = in none; clusters without
    a prediction are left out). Columns as SCORE_COLS: n, mean and sd of the observed IWI, R2 against the cell's own
    mean IWI (negative when the predictions miss the cell's level or spread), r2 (squared correlation of predicted and
    observed, which an offset does not lower; Yeh et al. 2020), RMSE, MAE and bias (mean of predicted - observed), in
    IWI points. R2 and r2 need min_n clusters (default MIN_N). Returns an array (n_cells, len(SCORE_COLS))."""
    min_n = MIN_N if min_n is None else min_n
    ok = np.isfinite(p) & (code >= 0)
    c, yy, pp = code[ok], y[ok], p[ok]
    e = pp - yy
    s = lambda w: np.bincount(c, weights=w, minlength=n_cells)[:n_cells]
    n = np.bincount(c, minlength=n_cells)[:n_cells].astype(float)
    sy, syy, sp, spp, spy = s(yy), s(yy * yy), s(pp), s(pp * pp), s(pp * yy)
    se, see, sae = s(e), s(e * e), s(np.abs(e))
    with np.errstate(invalid='ignore', divide='ignore'):
        my = sy / n
        sst = syy - sy * my                       # sum of squares around the cell's mean
        spp_c = spp - sp * sp / n
        cov = spy - sp * my
        enough = (n >= max(min_n, 2)) & (sst > 1e-9)
        r2 = np.where(enough, 1 - see / sst, np.nan)
        rr = np.where(enough & (spp_c > 1e-12), cov ** 2 / (sst * spp_c), np.nan)
        out = np.column_stack([n, my, np.sqrt(sst / (n - 1)), r2, rr, np.sqrt(see / n), sae / n, se / n])
    out[n == 0, 1:] = np.nan
    out[n < 2, 2] = np.nan
    return out


class Bootstrap:
    """Poisson bootstrap over 03's spatial groups (clusters closer than 9.5 km are one group, and a group lies in one
    fold): every resample gives every group a weight drawn from Poisson(1). The same weights serve every comparison, so
    all comparisons are paired, and a subset (a country, its rural clusters) is resampled with the groups it holds."""

    def __init__(self, unit, n_boot, seed):
        self.unit = np.asarray(unit, dtype=np.int64)
        self.n_boot = int(n_boot)
        n_units = int(self.unit.max()) + 1 if len(self.unit) else 0
        rng = np.random.default_rng(seed)
        self.W = rng.poisson(1.0, size=(self.n_boot, n_units)).astype(np.float64) if self.n_boot else None


def _pair_stats(T):
    """R2 of a and b, dR2 = R2(b) - R2(a), dRMSE and dMAE (b - a) from the sums T[..., :7]: n, sum y, sum y^2,
    SSE of a, SSE of b, sum |error of a|, sum |error of b|."""
    n, sy, syy, sa, sb, aa, ab = (T[..., k] for k in range(7))
    with np.errstate(invalid='ignore', divide='ignore'):
        sst = syy - sy * sy / n
        bad = (n < 2) | ~(sst > 1e-9)
        r2a = np.where(bad, np.nan, 1 - sa / sst)
        r2b = np.where(bad, np.nan, 1 - sb / sst)
        return r2a, r2b, r2b - r2a, np.sqrt(sb / n) - np.sqrt(sa / n), (ab - aa) / n


PAIR_COLS = ['n', 'R2 a', 'R2 b', 'dR2', 'dR2 low', 'dR2 high', 'P(dR2 > 0)', 'p', 'dRMSE', 'dRMSE low', 'dRMSE high',
             'dMAE', 'folds better', 'folds compared']


def compare(y, pa, pb, cells, boot, fold_code=None, draws=False):
    """Prediction column b against a on the clusters where both have a prediction, in every cell (name -> positions of
    its clusters). Per cell: n, R2 of a and b, dR2 = R2(b) - R2(a) (a and b share the clusters, so this is their
    difference in squared error over the cell's sum of squares), its 95 % bootstrap interval, P(dR2 > 0) and the
    two-sided p of the resamples, dRMSE and dMAE (b - a in IWI points; negative = b better), and in how many test folds
    b has the higher R2 (of the folds with at least MIN_FOLD_N of the cell's clusters). A cell with fewer than MIN_N
    common clusters gets n only. Returns {cell: dict with the keys of PAIR_COLS}; draws=True adds 'draws', the dR2 of
    every resample (the same resamples for every cell and comparison, so differences of draws are paired)."""
    both = np.isfinite(pa) & np.isfinite(pb)
    ea, eb = y - pa, y - pb
    Q = np.column_stack([np.ones(len(y)), y, y * y, ea * ea, eb * eb, np.abs(ea), np.abs(eb)])
    out = {}
    for name, pos in cells.items():
        pos = pos[both[pos]]
        rec = {'n': int(len(pos))}
        out[name] = rec
        if len(pos) < MIN_N:
            continue
        r2a, r2b, d, drmse, dmae = (float(v) for v in _pair_stats(Q[pos].sum(axis=0)))
        rec.update({'R2 a': r2a, 'R2 b': r2b, 'dR2': d, 'dRMSE': drmse, 'dMAE': dmae})
        if boot is not None and boot.n_boot:
            u, inv = np.unique(boot.unit[pos], return_inverse=True)
            inv = inv.ravel()
            S = np.column_stack([np.bincount(inv, weights=Q[pos, k], minlength=len(u)) for k in range(7)])
            _, _, bd, brmse, _ = _pair_stats(boot.W[:, u] @ S)
            ok = np.isfinite(bd)
            if draws:
                rec['draws'] = bd
            if ok.sum() >= 10:
                b = bd[ok]
                rec.update({'dR2 low': float(np.quantile(b, 0.025)), 'dR2 high': float(np.quantile(b, 0.975)),
                            'P(dR2 > 0)': float((b > 0).mean()),
                            'p': float(min(1.0, 2 * (1 + min((b <= 0).sum(), (b >= 0).sum())) / (len(b) + 1))),
                            'dRMSE low': float(np.nanquantile(brmse[ok], 0.025)),
                            'dRMSE high': float(np.nanquantile(brmse[ok], 0.975))})
        if fold_code is not None:
            f = fold_code[pos]
            keep = f >= 0
            Sf = np.column_stack([np.bincount(f[keep], weights=Q[pos[keep], k], minlength=len(FOLDS))[:len(FOLDS)]
                                  for k in range(7)])
            use = Sf[:, 0] >= MIN_FOLD_N
            fd = _pair_stats(Sf[use])[2]
            rec['folds better'] = int((fd > 0).sum())
            rec['folds compared'] = int(np.isfinite(fd).sum())
    return out


def bh(p):
    """Benjamini-Hochberg q-values of the p-values p (NaN stays NaN)."""
    p = np.asarray(p, dtype=float)
    q = np.full(len(p), np.nan)
    ok = np.flatnonzero(np.isfinite(p))
    if len(ok):
        order = ok[np.argsort(p[ok], kind='stable')]
        ranked = p[order] * len(ok) / np.arange(1, len(ok) + 1)
        q[order] = np.minimum(np.minimum.accumulate(ranked[::-1])[::-1], 1.0)
    return q


def add_q(frame, keys, where=None, split=None):
    """Column 'q': the Benjamini-Hochberg q-values of 'p' within every group of keys (rows of `where` only); `split`
    (one label per row, e.g. baseline_split) divides the groups further."""
    frame = frame.copy()
    frame['q'] = np.nan
    rows = frame.index if where is None else frame.index[np.asarray(where, bool)]
    by = list(keys)
    if split is not None:
        frame['_split'] = np.asarray(split)
        by.append('_split')
    for _, g in frame.loc[rows].groupby(by, sort=False):
        frame.loc[g.index, 'q'] = bh(g['p'].to_numpy(float))
    return frame.drop(columns='_split') if split is not None else frame


def baseline_split(frame):
    """'baseline' for the rows of a baseline model (BASELINE_MODELS; in a table of pairs: either model), 'main' for all
    others: add_q(..., split=baseline_split(frame)) gives the comparisons with a baseline model their own q-values, so
    those of the other models stay the same whether or not the baselines have run."""
    is_base = frame['model'].isin(BASELINE_MODELS).to_numpy()
    if 'other model' in frame:
        is_base = is_base | frame['other model'].isin(BASELINE_MODELS).to_numpy()
    return np.where(is_base, 'baseline', 'main')


def verdict(low, high):
    """'better' when the 95 % interval of a difference lies above 0, 'worse' below 0, 'n.s.' when it holds 0."""
    low, high = np.asarray(low, dtype=float), np.asarray(high, dtype=float)
    return np.where(low > 0, 'better', np.where(high < 0, 'worse', np.where(np.isfinite(low), 'n.s.', '')))


def fmt_delta(v, digits=3):
    """A signed difference without the leading zero: 0.0123 -> '+.012', -1.5 -> '-1.500', NaN -> '–'."""
    if v is None or not np.isfinite(v):
        return '–'
    return re.sub(r'^([+-])0\.', r'\1.', f'{v:+.{digits}f}')


def fmt_interval(d, low, high, digits=3):
    """'+.012 [+.004, +.020]' (without the interval when there is none)."""
    if d is None or not np.isfinite(d):
        return '–'
    if low is None or not np.isfinite(low):
        return fmt_delta(d, digits)
    return f'{fmt_delta(d, digits)} [{fmt_delta(low, digits)}, {fmt_delta(high, digits)}]'


def write_csv(frame, path, **kw):
    """Write a CSV file (gzip for .gz) under a temporary name, then rename it, so nobody reads half a file."""
    tmp = f'{path}.{uuid.uuid4().hex[:8]}.tmp'
    if path.endswith('.gz'):
        kw.setdefault('compression', 'gzip')
    frame.to_csv(tmp, index=False, **kw)
    os.replace(tmp, path)

In [ ]:
# Colors of the reference palette of the data-viz guide: chart ink, a blue -> red diverging scale with a gray midpoint
# for differences (blue: better than the reference), a one-hue blue scale for R2, and categorical slots 1 and 2 (blue,
# orange; checked for colour-vision deficiency) for rural and urban, with their own marker shapes.
INK, SECONDARY, MUTED, GRID, AXIS, SURFACE = '#0b0b0b', '#52514e', '#898781', '#e1e0d9', '#c3c2b7', '#fcfcfb'
SERIES_1    = '#2a78d6'                                    # categorical slot 1: a single series
AREA_COLOR  = {'rural': '#2a78d6', 'urban': '#eb6834'}
AREA_MARKER = {'rural': 'o', 'urban': 's'}
SEQUENTIAL  = LinearSegmentedColormap.from_list('r2_blue', ['#cde2fb', '#9ec5f4', '#6da7ec', '#3987e5', '#256abf',
                                                            '#184f95', '#0d366b'])
DIVERGING   = LinearSegmentedColormap.from_list('delta', ['#e34948', '#f0efec', '#2a78d6'])
for _cmap in (SEQUENTIAL, DIVERGING):
    _cmap.set_bad(SURFACE)
plt.rcParams.update({'figure.facecolor': SURFACE, 'axes.facecolor': SURFACE, 'savefig.facecolor': SURFACE,
                     'axes.edgecolor': AXIS, 'axes.labelcolor': SECONDARY, 'axes.titlecolor': INK,
                     'axes.titlesize': 10, 'axes.titlelocation': 'left', 'axes.labelsize': 9, 'font.size': 9,
                     'xtick.color': SECONDARY, 'ytick.color': SECONDARY, 'xtick.labelsize': 8,
                     'ytick.labelsize': 8, 'text.color': INK, 'legend.frameon': False, 'legend.fontsize': 8,
                     'axes.spines.top': False, 'axes.spines.right': False, 'grid.color': GRID,
                     'grid.linewidth': 0.8})


def text_color(rgba):
    """White on dark cells, ink on light ones."""
    r, g, b = rgba[:3]
    return 'white' if 0.2126 * r + 0.7152 * g + 0.0722 * b < 0.45 else INK


def delta_norm(values, floor=0.01, q=0.95):
    """A symmetric scale for differences: +-(the q quantile of |values|, at least floor), rounded up to 0.005."""
    v = np.abs(np.asarray(values, dtype=float))
    v = v[np.isfinite(v)]
    top = max(floor, float(np.quantile(v, q)) if len(v) else floor)
    top = np.ceil(top / 0.005) * 0.005
    return Normalize(-top, top)


def heatmap(ax, values, row_labels, col_labels, cmap, norm, labels=None, marks=None, bold=None, fontsize=6.5):
    """Coloured cells (rows x cols) with a thin surface gap. labels: text per cell (None: none); marks: cells that get a
    small dot (a significant difference, when there is no text); bold: cells whose text is bold. NaN cells get a muted
    dash."""
    values = np.asarray(values, dtype=float)
    im = ax.imshow(np.ma.masked_invalid(values), cmap=cmap, norm=norm, aspect='auto', interpolation='nearest')
    ax.set_xticks(np.arange(len(col_labels)))
    ax.set_xticklabels(col_labels, rotation=90)
    ax.set_yticks(np.arange(len(row_labels)))
    ax.set_yticklabels(row_labels)
    ax.set_xticks(np.arange(-0.5, len(col_labels)), minor=True)
    ax.set_yticks(np.arange(-0.5, len(row_labels)), minor=True)
    ax.grid(which='minor', color=SURFACE, linewidth=1.5)
    ax.grid(which='major', visible=False)
    ax.tick_params(which='both', length=0)
    for spine in ax.spines.values():
        spine.set_visible(False)
    for i in range(values.shape[0]):
        for j in range(values.shape[1]):
            v = values[i, j]
            if not np.isfinite(v):
                ax.text(j, i, '–', ha='center', va='center', fontsize=fontsize, color=MUTED)
                continue
            color = text_color(cmap(norm(v)))
            if labels is not None and labels[i][j]:
                ax.text(j, i, labels[i][j], ha='center', va='center', fontsize=fontsize, color=color,
                        fontweight='bold' if bold is not None and bold[i][j] else 'normal')
            elif marks is not None and marks[i][j]:
                ax.plot(j, i, marker='o', markersize=2.6, color=color, linestyle='none')
    return im


def forest(ax, y, d, low, high, color, marker='o', label=None):
    """Differences d with their 95 % intervals at the heights y: a filled marker where the interval excludes 0, an open
    one where it holds 0. Returns a handle for the legend."""
    d, low, high, y = (np.asarray(v, dtype=float) for v in (d, low, high, y))
    has = np.isfinite(low) & np.isfinite(high)
    ax.hlines(y[has], low[has], high[has], color=color, linewidth=1.4)
    sig = has & ((low > 0) | (high < 0))
    ok = np.isfinite(d)
    ax.scatter(d[ok & sig], y[ok & sig], s=34, marker=marker, color=color, edgecolors=SURFACE, linewidths=1.0,
               zorder=3)
    ax.scatter(d[ok & ~sig], y[ok & ~sig], s=34, marker=marker, facecolors=SURFACE, edgecolors=color,
               linewidths=1.3, zorder=3)
    return Line2D([], [], marker=marker, linestyle='none', color=color, markersize=6, label=label)


FILLED_OPEN = [Line2D([], [], marker='o', linestyle='none', color=SECONDARY, markersize=6,
                      label='filled: the 95 % interval excludes 0'),
               Line2D([], [], marker='o', linestyle='none', markerfacecolor=SURFACE, markeredgecolor=SECONDARY,
                      markersize=6, label='open: it holds 0')]


def save_figure(fig, name):
    """Save a figure as EVAL_DIR/<name>.png (written under a temporary name, then renamed) and show that file, so the
    notebook shows exactly the saved plot."""
    path = os.path.join(EVAL_DIR, f'{name}.png')
    fig.savefig(path + '.tmp.png', dpi=150, bbox_inches='tight')
    os.replace(path + '.tmp.png', path)
    plt.close(fig)
    display(Image(filename=path))
    print('->', path)


def grid_of(n, ncols):
    """Rows and columns of a grid of n panels with at most ncols columns, as even as possible (4 panels: 2 x 2)."""
    ncols = max(1, min(ncols, n))
    nrows = int(np.ceil(n / ncols))
    return nrows, int(np.ceil(n / nrows))

In [ ]:
def consensus(table, keys, extras=True):
    """Over the models of every group of keys (without SUMMARY_SKIP: the noise and the baseline models): models, median
    and mean dR2, models with dR2 > 0, models significantly better / worse (interval above / below 0), the model with
    the largest gain; the mean dR2 of the noise models (training noise) and the dR2 of every baseline model. With
    extras=False (the comparisons with a baseline model) the model with the smallest dR2 instead of the last two."""
    out = []
    baselines = [b for b in BASELINE_MODELS if b in set(table['model'])]
    for k, g in table.groupby(keys, sort=False):
        real = g[~g['model'].isin(SUMMARY_SKIP) & g['dR2'].notna()]
        noise = g[g['model'].isin(NOISE_MODELS) & g['dR2'].notna()]
        rec = dict(zip(keys, k if isinstance(k, tuple) else (k,)))
        rec.update({'n': int(g['n'].max()), 'models': len(real),
                    'median dR2': real['dR2'].median() if len(real) else np.nan,
                    'mean dR2': real['dR2'].mean() if len(real) else np.nan,
                    'models better': int((real['dR2'] > 0).sum()),
                    'sig. better': int((real['verdict'] == 'better').sum()),
                    'sig. worse': int((real['verdict'] == 'worse').sum()),
                    'best model': model_label(real.loc[real['dR2'].idxmax(), 'model']) if len(real) else '',
                    'best dR2': real['dR2'].max() if len(real) else np.nan})
        if extras:
            rec.update({'noise dR2': noise['dR2'].mean() if len(noise) else np.nan,
                        **{f'{b} dR2': g.loc[g['model'] == b, 'dR2'].mean() for b in baselines}})
        else:
            rec.update({'worst model': model_label(real.loc[real['dR2'].idxmin(), 'model']) if len(real) else '',
                        'worst dR2': real['dR2'].min() if len(real) else np.nan})
        out.append(rec)
    out = pd.DataFrame(out)
    return sort_runs(out.assign(model='')).drop(columns='model').reset_index(drop=True) if len(out) else out


def consensus_cell(r):
    """'+.012 (8/9 ▲3 ▼1)': median dR2, models better / models, significantly better (▲) and worse (▼)."""
    if not r['models'] or not np.isfinite(r['median dR2']):
        return ''
    return (f"{fmt_delta(r['median dR2'])} ({r['models better']}/{r['models']}"
            + (f" ▲{r['sig. better']}" if r['sig. better'] else '') + (f" ▼{r['sig. worse']}" if r['sig. worse'] else '')
            + ')')

In [ ]:
# ---- every model of 11-15 against the baseline models (BASELINE_MODELS: VGG-16-BN and the MLP on band statistics) ------
BM_AGAINST = ['same input', 'Landsat + NL']   # the run of the baseline model that a run is compared with: the one with
                                              # the same input (what the architecture adds over the baseline model) and
                                              # the one with Landsat + NL (what the architecture and the added bands add
                                              # together over the baseline model on the baseline input)
BM_COLS = ['start', 'family', 'model', 'combo', 'baseline model', 'against', 'baseline run']


def baseline_model_pairs(frame):
    """(run, run of the baseline model, baseline model, against) for every run of the models of 11-15 in frame and every
    baseline model with a run of the same start: against its run with the same input ('same input'; not for the
    baseline model itself) and against its run with Landsat + NL ('Landsat + NL'; for the baseline model itself this is
    its own gain from the input, as in the comparisons with Landsat + NL)."""
    t = frame[frame['family'] == 'ten_models']
    rid = {(r.model, r.combo, r.start): r.run_id for r in t.itertuples()}
    out = []
    for r in t.itertuples():
        for b in BASELINE_MODELS:
            same, base = rid.get((b, r.combo, r.start)), rid.get((b, BASELINE, r.start))
            if same is not None and r.model != b:
                out.append((r.run_id, same, b, 'same input'))
            if base is not None and base != r.run_id:
                out.append((r.run_id, base, b, 'Landsat + NL'))
    return out


def sort_bm(frame):
    """Rows in the order of start, baseline model, reference (against), model and input; rows that share these (the
    countries, the areas) keep their order."""
    keys = [(START_ORDER.index(s) if s in START_ORDER else 9, BASELINE_MODELS.index(b) if b in BASELINE_MODELS else 9,
             BM_AGAINST.index(a) if a in BM_AGAINST else 9, model_order(m), combo_order(c))
            for s, b, a, m, c in zip(frame['start'], frame['baseline model'], frame['against'], frame['model'],
                                     frame['combo'])]
    return frame.iloc[sorted(range(len(frame)), key=keys.__getitem__)].reset_index(drop=True)


def bm_label(b, against):
    """'VGG-16-BN, same input' or 'VGG-16-BN on Landsat + NL': a baseline model and the run it is compared with."""
    return f'{model_label(b)}, same input' if against == 'same input' else f'{model_label(b)} on Landsat + NL'


def bm_summary(table, keys):
    """consensus() of a table of comparisons with the baseline models over the ten models (without SUMMARY_SKIP), for
    every start, baseline model, reference and the other keys, in the order of sort_bm."""
    if not len(table):
        return pd.DataFrame(columns=['start', 'family', 'baseline model', 'against'] + list(keys) +
                            ['n', 'models', 'median dR2', 'mean dR2', 'models better', 'sig. better', 'sig. worse',
                             'best model', 'best dR2', 'worst model', 'worst dR2'])
    out = consensus(table.assign(family='ten_models'), ['start', 'family', 'baseline model', 'against'] + list(keys),
                    extras=False)
    return sort_bm(out.assign(model='')).drop(columns='model')

# 1. Clusters

All clusters of `train_clusters_folds.csv` (03), with their country, rural or urban flag (`rural` of the DHS table), fold, spatial group and IWI. Every run is checked against this table, and its IWI is the observed value of every comparison.

In [ ]:
clusters = pd.read_csv(FOLDS_CSV, dtype={'cluster_id': str})
clusters = clusters.rename(columns={'ClusterID': 'cluster_id', 'iwi_avg': 'iwi'})     # the older names, as in 02 and 03
assert clusters['cluster_id'].is_unique, f'a cluster_id appears twice in {FOLDS_CSV}'
assert TARGET in clusters, f'{FOLDS_CSV} has no column {TARGET}'
if 'nl_sensor' not in clusters:                                    # 01: DMSP up to 2013, VIIRS from 2014
    clusters['nl_sensor'] = np.where(clusters['year'] >= 2014, 'VIIRS', 'DMSP')
flag = (clusters['rural'].astype(str).str.strip().str.lower() if 'rural' in clusters
        else pd.Series('', index=clusters.index))
clusters['area'] = np.select([flag.isin(['true', '1', '1.0', 'r', 'rural']).to_numpy(),
                              flag.isin(['false', '0', '0.0', 'u', 'urban']).to_numpy()], ['rural', 'urban'], '')
group = (pd.to_numeric(clusters['group'], errors='coerce') if 'group' in clusters
         else pd.Series(np.nan, index=clusters.index)).to_numpy(dtype=float).copy()
no_group = ~np.isfinite(group)                                     # a cluster without a group is a group of its own
group[no_group] = np.max(group[~no_group], initial=-1) + 1 + np.arange(no_group.sum())
clusters['group'] = group.astype(np.int64)
clusters['fold'] = clusters['fold'].fillna('').astype(str)

N = len(clusters)
Y = clusters[TARGET].to_numpy(dtype=float)
ROW = pd.Series(np.arange(N), index=clusters['cluster_id'].to_numpy())
FOLD = clusters['fold'].to_numpy(dtype=str)
FOLD_CODE = pd.Categorical(FOLD, categories=FOLDS).codes.astype(np.int64)
COUNTRIES = sorted(clusters['country'].dropna().astype(str).unique())
BOOT = Bootstrap(clusters['group'].to_numpy(), BOOTSTRAP, SEED)
CONTINENT = f'all {len(COUNTRIES)} countries'
ALL = {CONTINENT: np.arange(N)}

print(f'{N} clusters, {len(COUNTRIES)} countries, {clusters["group"].nunique()} spatial groups'
      + (f' ({no_group.sum()} clusters without a group: one each)' if no_group.any() else '') + ' | folds:',
      clusters['fold'].value_counts().sort_index().to_dict(), '| rural / urban / no flag:',
      clusters['area'].replace('', 'no flag').value_counts().to_dict())
per_country = (clusters.assign(rural=clusters['area'].eq('rural'), urban=clusters['area'].eq('urban'),
                               viirs=clusters['nl_sensor'].eq('VIIRS'))
               .groupby('country').agg(clusters=('cluster_id', 'size'), rural=('rural', 'sum'),
                                       urban=('urban', 'sum'), VIIRS_period=('viirs', 'sum'),
                                       iwi_mean=(TARGET, 'mean'), iwi_sd=(TARGET, 'std'))
               .sort_values('clusters', ascending=False))
per_country.round(1)

# 2. Find the runs and collect their test predictions

The run sets are found by their folders (any start, variant and model found there is used):

| family | notebook | folders | input from |
|---|---|---|---|
| `ten_models` | 12, 11, 13 | `landsat_nl_runs/`, `landsat_nl_wavelet_runs/`, `landsat_nl_dft_runs/` + `<periods>_<start>/<model>/test_<fold>/` | the notebook |
| `ten_models` | 14 | `landsat_nl_peak_runs/<periods>_<start>_<alignment>_<input>/<model>/test_<fold>/` | `L+NL+peak:<alignment>/<input>` |
| `ten_models` | 15 | `landsat_nl_temporal_runs/<periods>_<start>_<position>/<model>/test_<fold>/` | `L+NL+TT:<position>` |
| `table2` | 04 | `table2_runs_fixednorm*/<row>/test_<fold>/` | the row |
| `table2` | 05 | `table2_resnet18_runs_fixednorm*/<mode>/<row>/test_<fold>/` | the row |
| `single` | 06, 07, 08 | `nl_resnet18_runs/<periods>_<start>/`, `landsat_resnet18_runs/<periods>_<start>/<row>/`, `landsat_vits16_runs/<periods>_<mode>/<row>/` + `test_<fold>/` | NL, the row |
| `ridge` | 09, 10 | `ridge_runs/<name>/<row>_test_predictions.csv` (prediction `pred NL + Landsat`) | the row |

A test fold counts when its folder holds `result.json` and `test_predictions.csv`. The earlier runs with standardised bands (`table2_runs/`) are not read. Every run is checked: its clusters must be in `train_clusters_folds.csv`, each predicted once, in its own fold, with the same IWI; otherwise it is listed with the problem and, with `DROP_STALE`, left out.

In [ ]:
def read_json(path):
    with open(path) as f:
        return json.load(f)


def result_start(results, default):
    """'pretrained' or 'scratch' from the result.json files of a run set: 'from_scratch' (04, 05, 08, 11-15) or
    'pretrained' (06, 07); runs written before these keys existed started from pretrained weights (default)."""
    found = set()
    for r in results:
        if 'from_scratch' in r:
            found.add('scratch' if r['from_scratch'] else 'pretrained')
        elif 'pretrained' in r:
            found.add('pretrained' if r['pretrained'] else 'scratch')
    if not found:
        return default
    return found.pop() if len(found) == 1 else 'mixed'


def run_sets(*parts):
    """{run-set folder: {fold: its test folder}} for the folders WORK_DIR/<parts>/test_<fold>/ that hold result.json and
    test_predictions.csv (parts may be glob patterns)."""
    sets = {}
    for res in sorted(glob.glob(os.path.join(WORK_DIR, *parts, 'test_*', 'result.json'))):
        test_dir = os.path.dirname(res)
        fold = os.path.basename(test_dir)[len('test_'):]
        if fold in FOLDS and os.path.exists(os.path.join(test_dir, 'test_predictions.csv')):
            sets.setdefault(os.path.dirname(test_dir), {})[fold] = test_dir
    return sets


def row_combo(key, base, head):
    """Input of a row key of 04/05 (base 'base') or 07-10 (base 'landsat'): 'base_wavelet6' -> head + '+wavelet',
    'landsat' -> head; None for an unknown key."""
    extra = {'ndvi': 'NDVI', 'wavelet6': 'wavelet', 'hfc_max': 'HFC_max', 'hfc_mean': 'HFC_mean',
             'hfc_min': 'HFC_min', 'lfc_max': 'LFC_max', 'lfc_mean': 'LFC_mean', 'lfc_min': 'LFC_min'}
    if key == base:
        return head
    rest = key[len(base) + 1:] if key.startswith(base + '_') else None
    return f'{head}+{extra[rest]}' if rest in extra else None


SOURCES, SKIPPED = [], []


def add_source(family, notebook, model, combo, path, folds, start_default):
    """A run set of 04-08 or 11-15: its start from result.json (05: also its model)."""
    results = [read_json(os.path.join(d, 'result.json')) for d in folds.values()]
    if notebook == '05':                                          # resnet18_finetune or resnet18_frozen
        model = next((r['model'] for r in results if r.get('model')), model)
    SOURCES.append({'family': family, 'notebook': notebook, 'model': model, 'combo': combo, 'path': path,
                    'folds': folds, 'start': result_start(results, start_default)})


def skip(path, why):
    SKIPPED.append({'folder': os.path.relpath(path, WORK_DIR), 'why': why})


TEN_ROOTS = {'12': 'landsat_nl_runs', '11': 'landsat_nl_wavelet_runs', '13': 'landsat_nl_dft_runs',
             '14': 'landsat_nl_peak_runs', '15': 'landsat_nl_temporal_runs'}
FIXED_INPUT = {'12': BASELINE, '11': f'{BASELINE}+wavelet', '13': f'{BASELINE}+DFT'}
if 'ten_models' in FAMILIES:
    for nb, root in TEN_ROOTS.items():
        for set_dir, folds in run_sets(root, f'{PERIODS}_*', '*').items():
            tag = os.path.basename(os.path.dirname(set_dir))[len(PERIODS) + 1:]     # <start>[_<variant>]
            start, _, variant = tag.partition('_')
            if start not in ('pretrained', 'scratch') or (nb in FIXED_INPUT) == bool(variant):
                skip(set_dir, f'not a run folder of {nb}: <periods>_<start>' + ('' if nb in FIXED_INPUT else
                                                                                   '_<variant>') + '/<model>')
                continue
            combo = FIXED_INPUT.get(nb) or (f'{BASELINE}+peak:' + variant.replace('_', '/', 1) if nb == '14'
                                            else f'{BASELINE}+TT:{variant}')
            add_source('ten_models', nb, os.path.basename(set_dir), combo, set_dir, folds, start)
if 'table2' in FAMILIES:
    for nb, parts, model in (('04', ('table2_runs_fixednorm*', '*'), 'vgg16'),
                             ('05', ('table2_resnet18_runs_fixednorm*', '*', '*'), None)):
        for set_dir, folds in run_sets(*parts).items():
            combo = row_combo(os.path.basename(set_dir), 'base', BASELINE)
            if combo is None:
                skip(set_dir, 'not a row of Table 2')
                continue
            mode = os.path.basename(os.path.dirname(set_dir))
            add_source('table2', nb, model or f"resnet18_{'frozen' if mode.startswith('frozen') else 'finetune'}",
                       combo, set_dir, folds, 'pretrained')
if 'single' in FAMILIES or 'ridge' in FAMILIES:                   # the ridge needs them to find its start
    for nb, parts in (('06', ('nl_resnet18_runs', f'{PERIODS}_*')), ('07', ('landsat_resnet18_runs', f'{PERIODS}_*', '*')),
                      ('08', ('landsat_vits16_runs', f'{PERIODS}_*', '*'))):
        for set_dir, folds in run_sets(*parts).items():
            run_root = set_dir if nb == '06' else os.path.dirname(set_dir)
            tag = os.path.basename(run_root)[len(PERIODS) + 1:]   # imagenet, scratch; 08: finetune, frozen, ...
            combo = 'NL' if nb == '06' else row_combo(os.path.basename(set_dir), 'landsat', 'L')
            if combo is None:
                skip(set_dir, f'not a row of {nb}')
                continue
            model = {'06': 'nl_resnet18', '07': 'landsat_resnet18'}.get(nb, f'landsat_vits16_{tag}')
            add_source('single', nb, model, combo, set_dir, folds, 'scratch' if 'scratch' in tag else 'pretrained')
if 'ridge' in FAMILIES:
    for path in sorted(glob.glob(os.path.join(WORK_DIR, 'ridge_runs', '*', '*_test_predictions.csv'))):
        name = os.path.basename(os.path.dirname(path))
        combo = row_combo(os.path.basename(path)[:-len('_test_predictions.csv')], 'landsat', BASELINE)
        if combo is None:
            skip(path, 'not a row of 09 or 10')
            continue
        SOURCES.append({'family': 'ridge', 'notebook': '', 'model': f'ridge:{name}', 'combo': combo, 'path': path,
                        'folds': None, 'start': None, 'name': name})
print(f'{len(SOURCES)} run sets found', pd.Series([s['family'] for s in SOURCES]).value_counts().to_dict() if SOURCES
      else '', f'| {len(SKIPPED)} folders skipped')
assert SOURCES, (f'no finished run (test_<fold>/result.json and test_predictions.csv) of the families {FAMILIES} under '
                 f'{WORK_DIR} (periods {PERIODS}): run 04-15 first')
if SKIPPED:
    display(pd.DataFrame(SKIPPED))

In [ ]:
def read_predictions(src):
    """The test predictions of a run set: cluster_id, test_fold, y_true, y_pred (a ridge run: also the IWI that the two
    networks of its features predicted, columns 'pred ... network (..)')."""
    if src['family'] == 'ridge':
        d = pd.read_csv(src['path'], dtype={'cluster_id': str, 'fold': str})
        out = pd.DataFrame({'cluster_id': d['cluster_id'], 'test_fold': d['fold'].astype(str), 'y_true': d[TARGET],
                            'y_pred': d['pred NL + Landsat']})
        for c in d.columns:
            if c.startswith('pred ') and ' network ' in c:
                out[c] = d[c]
        return out
    parts = [pd.read_csv(os.path.join(test_dir, 'test_predictions.csv'), usecols=['cluster_id', 'y_true', 'y_pred'],
                         dtype={'cluster_id': str}).assign(test_fold=fold) for fold, test_dir in src['folds'].items()]
    return pd.concat(parts, ignore_index=True)


def place(d, column='y_pred'):
    """A prediction column over all clusters (NaN where the run has no test prediction) and the problems found: test
    clusters that are not in FOLDS_CSV, predicted twice, in another fold of FOLDS_CSV, or with another IWI."""
    pos = ROW.reindex(d['cluster_id'].to_numpy()).to_numpy(dtype=float)
    known = np.isfinite(pos)
    p = pos[known].astype(np.int64)
    problems = {'not in FOLDS_CSV': int((~known).sum()), 'predicted twice': int(d['cluster_id'].duplicated().sum()),
                'other fold': int((FOLD[p] != d['test_fold'].to_numpy(dtype=str)[known]).sum()),
                'other IWI': int((np.abs(Y[p] - d['y_true'].to_numpy(dtype=float)[known]) > 0.01).sum())}
    col = np.full(N, np.nan)
    col[p] = d[column].to_numpy(dtype=float)[known]
    return col, problems


def same_predictions(a, b):
    """Two prediction columns agree (up to the rounding of the CSV files) on at least MIN_N common clusters."""
    m = np.isfinite(a) & np.isfinite(b)
    return m.sum() >= MIN_N and float(np.max(np.abs(a[m] - b[m]))) < 0.05


t0 = time.time()
records, COLS = [], []                                            # one entry per run set, in reading order
for src in sorted(SOURCES, key=lambda s: s['family'] == 'ridge'):  # the ridge last: it needs the runs of 06-08
    d = read_predictions(src)
    col, problems = place(d)
    note = ''
    if src['family'] == 'ridge':                                  # start: that of the networks of its features
        nets = [c for c in d.columns if c.startswith('pred ') and ' network ' in c]
        src['notebook'] = '10' if any('(10)' in c for c in nets) else '09'
        starts = []
        for c in nets:
            net_col, _ = place(d, c)
            if c.startswith('pred NL network'):
                cand = [k for k, r in enumerate(records) if r['model'] == 'nl_resnet18']
            else:
                ls_combo = 'L' + src['combo'][len(BASELINE):]     # the same row of 07 (09) or 08 (10)
                cand = [k for k, r in enumerate(records) if r['combo'] == ls_combo and
                        (r['model'] == 'landsat_resnet18' if src['notebook'] == '09'
                         else r['model'].startswith('landsat_vits16_'))]
            found = sorted({records[k]['start'] for k in cand if not records[k]['stale']
                            and same_predictions(net_col, COLS[k])})
            starts.append(found[0] if len(found) == 1 else None)
        if len(starts) == 2 and None not in starts:
            src['start'] = starts[0] if starts[0] == starts[1] else 'mixed'
            note = 'start: that of the networks of its features'
        else:
            src['start'] = RIDGE_START.get(src['name'], 'unknown')
            note = ('networks of its features not found: start from RIDGE_START' if src['name'] in RIDGE_START else
                    'networks of its features not found among the runs of 06-08: start unknown (set RIDGE_START)')
    folds_found = sorted(set(d['test_fold'].astype(str)) & set(FOLDS))
    stale = sum(problems.values()) > 0
    sensors = clusters.loc[np.isfinite(col), 'nl_sensor'].value_counts()
    records.append({'run_id': f"{src['family']}|{src['model']}|{src['combo']}|{src['start']}",
                    'family': src['family'], 'notebook': src['notebook'], 'model': src['model'],
                    'model_label': model_label(src['model']), 'combo': src['combo'],
                    'combo_label': combo_label(src['combo']), 'start': src['start'], 'folds': ''.join(folds_found),
                    'n_folds': len(folds_found), 'n_test': int(np.isfinite(col).sum()),
                    'clusters': '_'.join(s for s in ('DMSP', 'VIIRS') if s in sensors.index),
                    'problems': '; '.join(f'{k}: {v}' for k, v in problems.items() if v), 'stale': stale,
                    'used': not (stale and DROP_STALE) and (src['family'] != 'single' or 'single' in FAMILIES),
                    'note': note, 'path': os.path.relpath(src['path'], WORK_DIR)})
    COLS.append(col)

runs = pd.DataFrame(records, columns=['run_id', 'family', 'notebook', 'model', 'model_label', 'combo', 'combo_label',
                                      'start', 'folds', 'n_folds', 'n_test', 'clusters', 'problems', 'stale', 'used',
                                      'note', 'path'])
runs['used'], runs['stale'] = runs['used'].astype(bool), runs['stale'].astype(bool)
runs.loc[(runs['family'] == 'single') & ('single' not in FAMILIES), 'note'] = 'read only to find the start of the ridge'
# the same family, model, input and start found twice (for example in two run folders of 04): the one with more clusters
for run_id, g in runs[runs['used']].groupby('run_id'):
    if len(g) > 1:
        keep = g['n_test'].idxmax()
        runs.loc[g.index.drop(keep), 'used'] = False
        runs.loc[g.index.drop(keep), 'note'] = f"duplicate of {runs.at[keep, 'path']}"
COLUMNS = {runs.at[k, 'run_id']: COLS[k] for k in runs.index[runs['used']]}
runs = sort_runs(runs).reset_index(drop=True)
USED = runs[runs['used']].reset_index(drop=True)
assert len(USED), 'every run set found has problems (see below and eval_runs.csv): nothing to evaluate'
print(f'{len(runs)} run sets read in {time.time() - t0:.0f} s: {len(USED)} used, '
      f"{int(runs['stale'].sum())} with problems{' (left out)' if DROP_STALE else ''}")

In [ ]:
META = ['cluster_id', 'country', 'year', 'area', 'rural', 'nl_sensor', 'fold', 'group', 'lat', 'lon', TARGET]
wide = pd.concat([clusters[[c for c in META if c in clusters]].reset_index(drop=True),
                  pd.DataFrame({r: COLUMNS[r] for r in USED['run_id']}, index=range(N))], axis=1)
write_csv(runs, os.path.join(EVAL_DIR, 'eval_runs.csv'))
write_csv(wide, os.path.join(EVAL_DIR, 'eval_test_predictions.csv.gz'), float_format='%.7g')
print('->', os.path.join(EVAL_DIR, 'eval_runs.csv'))
print('->', os.path.join(EVAL_DIR, 'eval_test_predictions.csv.gz'), f'({N} clusters x {len(USED)} runs)')

# what was found: the test folds of every model and input of 11-15 (the ten models and the baseline models), per start
for start in [s for s in START_ORDER if s in set(runs['start'])]:
    t = runs[(runs['family'] == 'ten_models') & (runs['start'] == start)]
    if len(t):
        m = t.assign(cell=np.where(t['used'], t['folds'], t['folds'] + ' (not used)')).pivot_table(
            index='model', columns='combo', values='cell', aggfunc=' / '.join)
        m = m.reindex(index=sorted(m.index, key=model_order), columns=sorted(m.columns, key=combo_order)).fillna('·')
        print(f'\nthe models of 11-15, {start}: test folds done (· = no run)')
        display(m.rename(columns=short_combo))
other = runs[runs['family'] != 'ten_models']
if len(other):
    print('the other families:')
    display(other[['family', 'notebook', 'model', 'combo', 'start', 'folds', 'n_test', 'clusters', 'used', 'note',
                   'path']])
bad = runs[runs['stale']]
if len(bad):
    print('run sets whose test predictions do not agree with FOLDS_CSV' +
          (' (left out: DROP_STALE)' if DROP_STALE else ' (used anyway: DROP_STALE = False)') +
          '. They were made with other folds or clusters: train them again in new run folders.')
    display(bad[['run_id', 'folds', 'n_test', 'problems', 'path']])

# 3. Scores of every run on the whole continent

For every used run, on its own test clusters: R² and r² of the pooled test predictions, the mean and sd of the R² of its single test folds, RMSE, MAE and bias. Then, for every run whose family, model and start have a baseline run (`L+NL`), the paired comparison with that baseline on the clusters both have: `dR2` with its interval, `p`, `q` (within start and family), `dRMSE`, `dMAE` and `folds better`. Everything goes to `overall_metrics.csv`; sections 4 and 5 show it.

In [ ]:
ZERO = np.zeros(N, dtype=np.int64)
records = []
for r in USED.itertuples():
    p = COLUMNS[r.run_id]
    rec = {'start': r.start, 'family': r.family, 'model': r.model, 'combo': r.combo, 'notebook': r.notebook,
           'folds': r.folds, 'clusters': r.clusters,
           **dict(zip(SCORE_COLS, cell_scores(Y, p, ZERO, 1)[0]))}
    fr2 = cell_scores(Y, p, FOLD_CODE, len(FOLDS), min_n=MIN_FOLD_N)[:, 3]
    fr2 = fr2[np.isfinite(fr2)]
    rec['fold R2 mean'] = fr2.mean() if len(fr2) else np.nan
    rec['fold R2 sd'] = fr2.std(ddof=1) if len(fr2) > 1 else np.nan
    rec['run_id'] = r.run_id
    records.append(rec)
overall = pd.DataFrame(records)

BASE_RUN = {(r.family, r.model, r.start): r.run_id for r in USED.itertuples() if r.combo == BASELINE}
vs = []
for r in USED.itertuples():
    b = BASE_RUN.get((r.family, r.model, r.start))
    if r.combo == BASELINE or b is None:
        vs.append({})
        continue
    res = compare(Y, COLUMNS[b], COLUMNS[r.run_id], ALL, BOOT, FOLD_CODE)[CONTINENT]
    vs.append({'baseline run': b, 'paired n': res['n'], 'R2 baseline (paired)': res.get('R2 a', np.nan),
               **{k: res.get(k, np.nan) for k in PAIR_COLS[3:]}})
overall = pd.concat([overall, pd.DataFrame(vs, index=overall.index)], axis=1)
for c in ['baseline run', 'paired n', 'R2 baseline (paired)'] + PAIR_COLS[3:]:
    if c not in overall:
        overall[c] = np.nan
overall = add_q(overall, ['start', 'family'], where=overall['p'].notna(), split=baseline_split(overall))
overall['vs baseline'] = verdict(overall['dR2 low'], overall['dR2 high'])
overall = sort_runs(overall).reset_index(drop=True)
write_csv(overall, os.path.join(EVAL_DIR, 'overall_metrics.csv'))
print(f'{len(overall)} runs scored, {overall["dR2"].notna().sum()} of them against their baseline run ->',
      os.path.join(EVAL_DIR, 'overall_metrics.csv'))

# 4. Leaderboards: pretrained with pretrained, scratch with scratch

One table per start and set of test clusters (the Table 2 runs of 04/05 have only the VIIRS clusters, so their R² is not comparable with the others), sorted by the R² of the pooled test predictions. `R2 folds` is the mean ± sd of the R² of the single test folds. A run with fewer than five folds (`folds`) has fewer test clusters (`n`); section 6 compares models on the clusters they share. The plot shows the R² of every model and input of 11–15 (the ten models and the baseline models), one panel per start.

In [ ]:
def leaderboard(t):
    out = pd.DataFrame({'model': t['model'].map(model_label), 'input': t['combo'].map(short_combo),
                        'family': t['family'], 'folds': t['folds'], 'n': t['n'].astype(int),
                        'R2': t['R2'], 'R2 folds': [f'{m:.3f} ± {s:.3f}' if np.isfinite(s) else f'{m:.3f}'
                                                    for m, s in zip(t['fold R2 mean'], t['fold R2 sd'])],
                        'r2': t['r2'], 'RMSE': t['RMSE'], 'MAE': t['MAE'], 'bias': t['bias'],
                        'dR2 vs baseline [95 %]': [fmt_interval(d, lo, hi) if np.isfinite(d) else ''
                                                   for d, lo, hi in zip(t['dR2'], t['dR2 low'], t['dR2 high'])],
                        'vs baseline': t['vs baseline']})
    out = out.sort_values('R2', ascending=False, kind='stable')
    out.insert(0, 'rank', np.arange(1, len(out) + 1))
    return out.reset_index(drop=True)


for start in [s for s in START_ORDER if s in set(overall['start'])]:
    for cl in sorted(set(overall.loc[overall['start'] == start, 'clusters']), key=lambda c: (c != PERIODS, c)):
        t = overall[(overall['start'] == start) & (overall['clusters'] == cl)]
        print(f'\n{start.upper()} runs, test clusters of the {cl.replace("_", " + ")} period'
              f'{"s" if "_" in cl else ""}: {len(t)} runs')
        display(leaderboard(t).round(3))

In [ ]:
def r2_heatmaps(table, family, name, title):
    """R2 of every model (rows) and input (columns) of a family, one panel per start."""
    t = table[table['family'] == family]
    starts = [s for s in START_ORDER if s in set(t['start'])]
    if not len(t):
        print(f'no runs of {family}')
        return
    combos = sorted(set(t['combo']), key=combo_order)
    models = sorted(set(t['model']), key=model_order)
    v = t['R2'].to_numpy(dtype=float)
    v = v[np.isfinite(v)]
    lo = np.floor(np.quantile(v, 0.1) * 20) / 20 if len(v) else 0.0
    hi = max(np.ceil(v.max() * 20) / 20, lo + 0.05) if len(v) else 1.0
    norm = Normalize(lo, hi)
    fig, axes = plt.subplots(1, len(starts), figsize=(0.75 * len(combos) * len(starts) + 2.6, 0.36 * len(models) + 2.6),
                             squeeze=False, constrained_layout=True)
    for ax, start in zip(axes[0], starts):
        m = t[t['start'] == start].pivot(index='model', columns='combo', values='R2').reindex(index=models,
                                                                                            columns=combos)
        labels = [[f'{x:.3f}' if np.isfinite(x) else '' for x in row] for row in m.to_numpy(dtype=float)]
        im = heatmap(ax, m.to_numpy(dtype=float), [model_label(k) for k in models], [short_combo(c) for c in combos],
                     SEQUENTIAL, norm, labels=labels)
        ax.set_title(f'{start}', fontsize=10)
        if ax is not axes[0][0]:
            ax.set_yticklabels([])
    fig.colorbar(im, ax=list(axes[0]), shrink=0.7, label='R² of the pooled test predictions',
                 extend='min' if np.nanmin(t['R2']) < lo else 'neither')
    fig.suptitle(title, x=0.01, ha='left', fontsize=11)
    save_figure(fig, name)


r2_heatmaps(overall, 'ten_models', 'overall_r2_ten_models',
            'The models of 11-15: R² of the pooled test predictions, all clusters')
for fam in ('table2', 'ridge'):
    if fam in set(overall['family']):
        r2_heatmaps(overall, fam, f'overall_r2_{fam}', f'{FAMILY_LABEL[fam].capitalize()}: R² of the pooled test '
                    'predictions')

# 5. Every input against the baseline (Landsat + NL), on the whole continent

For every model and start, each input against the run of the same model and start with Landsat + NL alone, on the test clusters both have: ΔR² with its 95 % interval (positive: the added bands help). `q` is the Benjamini–Hochberg q-value over all models and inputs of a start and family (over the baseline models on their own). `satclip_only` reads no band, so its ΔR² shows the size of the training noise. The baseline models (VGG-16-BN, MLP) are shown like every other model; the summary over the models at the end leaves them out (with `satclip_only`) and shows their ΔR² on their own. The plots show one panel per input (models of 11–15) or per model (Table 2, ridge): filled markers where the interval excludes 0.

In [ ]:
def delta_table(t, rows, cols, row_order, col_order):
    """ΔR² [95 %] and the verdict of every row x column of a table of paired comparisons."""
    t = t.assign(cell=[fmt_interval(d, lo, hi) + {'better': ' ▲', 'worse': ' ▼'}.get(v, '')
                       for d, lo, hi, v in zip(t['dR2'], t['dR2 low'], t['dR2 high'], t['vs baseline'])])
    m = t.pivot(index=rows, columns=cols, values='cell')
    return m.reindex(index=sorted(m.index, key=row_order), columns=sorted(m.columns, key=col_order)).fillna('')


def forest_figure(t, panel_key, row_key, name, title, x_label='ΔR² against Landsat + NL (95 % interval)'):
    """ΔR² with its interval: one panel per value of panel_key, one row per value of row_key."""
    t = t[t['dR2'].notna()]
    if not len(t):
        print('nothing to plot for', name)
        return
    order = {'combo': combo_order, 'model': model_order}
    label = {'combo': short_combo, 'model': model_label}
    panels = sorted(set(t[panel_key]), key=order[panel_key])
    rows = sorted(set(t[row_key]), key=order[row_key])
    nr, nc = grid_of(len(panels), 4)
    fig, axes = plt.subplots(nr, nc, figsize=(3.4 * nc + 1.6, (0.27 * len(rows) + 1.2) * nr + 0.6), sharex=True,
                             sharey=True, squeeze=False, constrained_layout=True)
    lo = np.nanmin(np.r_[t['dR2 low'].to_numpy(float), t['dR2'].to_numpy(float), 0.0])
    hi = np.nanmax(np.r_[t['dR2 high'].to_numpy(float), t['dR2'].to_numpy(float), 0.0])
    pad = 0.05 * (hi - lo) + 1e-3
    yy = np.arange(len(rows))[::-1]
    for ax, panel in zip(axes.flat, panels):
        g = t[t[panel_key] == panel].set_index(row_key).reindex(rows)
        ax.axvline(0, color=SECONDARY, linewidth=0.8)
        forest(ax, yy, g['dR2'], g['dR2 low'], g['dR2 high'], SERIES_1)
        ax.set_yticks(yy)
        ax.set_yticklabels([label[row_key](k) for k in rows])
        ax.set_xlim(lo - pad, hi + pad)
        ax.xaxis.set_major_locator(plt.MaxNLocator(5))
        ax.grid(axis='x')
        ax.set_axisbelow(True)
        ax.set_title(label[panel_key](panel), fontsize=9)
    for ax in axes.flat[len(panels):]:
        ax.axis('off')
    for k, ax in enumerate(axes.flat[:len(panels)]):
        if k + nc >= len(panels):                                  # the lowest panel of its column
            ax.set_xlabel(x_label)
            ax.xaxis.set_tick_params(labelbottom=True)
    fig.legend(handles=FILLED_OPEN, loc='outside lower center', ncol=2)
    fig.suptitle(title, x=0.01, ha='left', fontsize=11)
    save_figure(fig, name)


have = overall[overall['dR2'].notna()]
for start in [s for s in START_ORDER if s in set(have['start'])]:
    for fam in [f for f in FAMILY_ORDER if f in set(have.loc[have['start'] == start, 'family'])]:
        t = have[(have['start'] == start) & (have['family'] == fam)]
        if fam == 'ten_models':
            print(f'\n{start.upper()}, {FAMILY_LABEL[fam]}: ΔR² against Landsat + NL [95 %] (▲ better, ▼ worse)')
            display(delta_table(t, 'model', 'combo', model_order, combo_order).rename(index=model_label,
                                                                                       columns=short_combo))
            forest_figure(t, 'combo', 'model', f'overall_vs_baseline_{start}_{fam}',
                          f'{start.capitalize()}: every input of the models of 11-15 against Landsat + NL, whole continent')
        else:
            print(f'\n{start.upper()}, {FAMILY_LABEL[fam]}: ΔR² against Landsat + NL [95 %] (▲ better, ▼ worse)')
            display(delta_table(t, 'combo', 'model', combo_order, model_order).rename(index=short_combo,
                                                                                       columns=model_label))
            forest_figure(t, 'model', 'combo', f'overall_vs_baseline_{start}_{fam}',
                          f'{start.capitalize()}: {FAMILY_LABEL[fam]}, every row against Landsat + NL')
summary = (have[~have['model'].isin(SUMMARY_SKIP)].groupby(['start', 'family', 'combo'], sort=False)
           .agg(models=('dR2', 'size'), median_dR2=('dR2', 'median'), better=('vs baseline', lambda v: (v == 'better').sum()),
                worse=('vs baseline', lambda v: (v == 'worse').sum())).reset_index())
for b in [b for b in BASELINE_MODELS if b in set(have['model'])]:         # every baseline model on its own
    summary = summary.merge(have.loc[have['model'] == b, ['start', 'family', 'combo', 'dR2']]
                            .rename(columns={'dR2': f'{b} dR2'}), on=['start', 'family', 'combo'], how='left')
if len(summary):
    print('per input, over the models (without', SUMMARY_SKIP, '): median ΔR², models better and worse than Landsat + NL'
          + ('; the ΔR² of the baseline models on their own' if set(BASELINE_MODELS) & set(have['model']) else ''))
    display(sort_runs(summary.assign(model='')).drop(columns='model').round(4))

# 6. Models against each other: pretrained with pretrained, scratch with scratch

For every start, family and input, every pair of models on the test clusters both have: ΔR² = R²(model) − R²(other model) with its interval, `p` and `q` (Benjamini–Hochberg over the pairs of the same start, family and input; the pairs with a baseline model on their own). `overall_best_models.csv` names the best model of every start, family and input (highest R²) and the models that cannot be told apart from it (interval of the difference holds 0). The plot shows the matrices of the models of 11–15 for the baseline and the main inputs (`MAIN_INPUTS`): row model minus column model, blue where the row model is better, bold where the interval excludes 0 (the colour scale is that of the ten models; the rows and columns of the baseline models are drawn with it). The last part shows every model against the baseline models for every input.

In [ ]:
pair_rows = []
for (start, fam, combo), g in USED.groupby(['start', 'family', 'combo'], sort=False):
    models = sorted(g['model'], key=model_order)
    rid = dict(zip(g['model'], g['run_id']))
    for i, a in enumerate(models):
        for b in models[i + 1:]:
            res = compare(Y, COLUMNS[rid[b]], COLUMNS[rid[a]], ALL, BOOT, FOLD_CODE)[CONTINENT]   # a minus b
            rec = {'start': start, 'family': fam, 'combo': combo, 'model': a, 'other model': b, 'n': res['n'],
                   'R2 model': res.get('R2 b', np.nan), 'R2 other': res.get('R2 a', np.nan),
                   **{k: res.get(k, np.nan) for k in PAIR_COLS[3:]}}
            pair_rows.append(rec)
pairs = pd.DataFrame(pair_rows, columns=['start', 'family', 'combo', 'model', 'other model', 'n', 'R2 model',
                                         'R2 other'] + PAIR_COLS[3:])
if len(pairs):
    pairs = add_q(pairs, ['start', 'family', 'combo'], where=pairs['p'].notna(), split=baseline_split(pairs))
    mirror = pairs.rename(columns={'model': 'other model', 'other model': 'model', 'R2 model': 'R2 other',
                                   'R2 other': 'R2 model', 'dR2 low': 'dR2 high', 'dR2 high': 'dR2 low',
                                   'dRMSE low': 'dRMSE high', 'dRMSE high': 'dRMSE low'})
    for c in ('dR2', 'dR2 low', 'dR2 high', 'dRMSE', 'dRMSE low', 'dRMSE high', 'dMAE'):
        mirror[c] = -mirror[c]
    mirror['P(dR2 > 0)'] = 1 - mirror['P(dR2 > 0)']
    mirror['folds better'] = mirror['folds compared'] - mirror['folds better']
    pairs = pd.concat([pairs, mirror[pairs.columns]], ignore_index=True)
    pairs['verdict'] = verdict(pairs['dR2 low'], pairs['dR2 high'])
    pairs = sort_runs(pairs).reset_index(drop=True)
write_csv(pairs, os.path.join(EVAL_DIR, 'overall_pairwise_models.csv'))

best_rows = []
for (start, fam, combo), g in overall.groupby(['start', 'family', 'combo'], sort=False):
    if len(g) < 2:
        continue
    best = g.loc[g['R2'].idxmax()]
    p = pairs[(pairs['start'] == start) & (pairs['family'] == fam) & (pairs['combo'] == combo) &
              (pairs['model'] == best['model'])]
    tied = sorted(p.loc[p['verdict'] == 'n.s.', 'other model'], key=model_order)
    best_rows.append({'start': start, 'family': fam, 'combo': combo, 'model': best['model'], 'models': len(g),
                      'best model': model_label(best['model']), 'R2': best['R2'],
                      'tied with the best': ', '.join(model_label(m) for m in tied),
                      'clearly worse': int((p['verdict'] == 'better').sum())})
best_models = pd.DataFrame(best_rows)
if len(best_models):
    best_models = sort_runs(best_models).drop(columns='model').reset_index(drop=True)
write_csv(best_models, os.path.join(EVAL_DIR, 'overall_best_models.csv'))
print(f'{len(pairs) // 2} pairs of models compared -> {os.path.join(EVAL_DIR, "overall_pairwise_models.csv")}')
if len(best_models):
    print('the best model of every start, family and input, and the models that cannot be told apart from it:')
    display(best_models.assign(combo=best_models['combo'].map(short_combo)).round(3))
else:
    print('no start, family and input with two or more models')

In [ ]:
def pairwise_figure(start):
    t = pairs[(pairs['start'] == start) & (pairs['family'] == 'ten_models')]
    combos = [c for c in [BASELINE] + MAIN_INPUTS if c in set(t['combo'])]
    if not combos:
        return
    models = sorted(set(t['model']) | set(t['other model']), key=model_order)
    norm = delta_norm(t.loc[~t['model'].isin(SUMMARY_SKIP) & ~t['other model'].isin(SUMMARY_SKIP), 'dR2'], floor=0.02)
    nr, nc = grid_of(len(combos), 3)
    size = 0.42 * len(models) + 1.2
    fig, axes = plt.subplots(nr, nc, figsize=(size * nc + 1.2, size * nr + 0.4), squeeze=False,
                             constrained_layout=True)
    for ax, combo in zip(axes.flat, combos):
        g = t[t['combo'] == combo]
        m = g.pivot(index='model', columns='other model', values='dR2').reindex(index=models, columns=models)
        sig = g.assign(s=g['verdict'].isin(['better', 'worse']).astype(float)).pivot(
            index='model', columns='other model', values='s').reindex(index=models, columns=models).fillna(0) > 0.5
        labels = [[fmt_delta(x) if np.isfinite(x) else '' for x in row] for row in m.to_numpy(dtype=float)]
        im = heatmap(ax, m.to_numpy(dtype=float), [model_label(k) for k in models], [model_label(k) for k in models],
                     DIVERGING, norm, labels=labels, bold=sig.to_numpy(dtype=bool), fontsize=5.5)
        ax.set_title(short_combo(combo), fontsize=9)
    for ax in axes.flat[len(combos):]:
        ax.axis('off')
    fig.colorbar(im, ax=list(axes.flat), shrink=0.5, extend='both',
                 label='R²(row model) − R²(column model); bold: 95 % interval excludes 0')
    fig.suptitle(f'{start.capitalize()}: the models of 11-15 against each other, whole continent', x=0.01, ha='left',
                 fontsize=11)
    save_figure(fig, f'overall_pairwise_{start}')


for start in [s for s in START_ORDER if len(pairs) and s in set(pairs['start'])]:
    pairwise_figure(start)

## Every model against the baseline models

The two baseline models of 11–15 (`BASELINE_MODELS`: VGG-16-BN and the MLP on band statistics, trained with the same inputs, folds and recipe as the ten models) are the reference for every model of 11–15, with the same start, in two ways (`against`):

* **same input**: ΔR² = R²(model with input I) − R²(baseline model with input I): what the architecture adds over the baseline model;
* **Landsat + NL**: ΔR² = R²(model with input I) − R²(baseline model with Landsat + NL): what the architecture and the added bands add together over the baseline model on the baseline input. The row of the baseline model itself is its own gain from the input (as in section 5).

Every comparison is paired on the test clusters both runs have, with its 95 % interval, `p` and `q` (Benjamini–Hochberg over the models and inputs of a start, baseline model and reference); ▲: the model is better than the baseline model (interval above 0), ▼: worse. `overall_vs_baseline_models.csv` holds every comparison; `overall_vs_baseline_models_summary.csv` the summary over the ten models (without `satclip_only` and the baseline models): **median ΔR² (models better / models; ▲ significantly better, ▼ worse)**, and the weakest model. The plot `overall_vs_baseline_models_<start>.png` shows every model and input: one row of panels per reference, one column per baseline model.

In [ ]:
INFO = {r.run_id: r for r in USED.itertuples()}
BM_PAIRS = baseline_model_pairs(USED)
rows = []
for run_id, ref_id, bm, against in BM_PAIRS:
    res = compare(Y, COLUMNS[ref_id], COLUMNS[run_id], ALL, BOOT, FOLD_CODE)[CONTINENT]
    r = INFO[run_id]
    rows.append({'start': r.start, 'family': r.family, 'model': r.model, 'combo': r.combo, 'baseline model': bm,
                 'against': against, 'baseline run': ref_id, **{k: res.get(k, np.nan) for k in PAIR_COLS}})
vs_bm = pd.DataFrame(rows, columns=BM_COLS + PAIR_COLS).rename(columns={'R2 a': 'R2 baseline model', 'R2 b': 'R2'})
vs_bm = add_q(vs_bm, ['start', 'baseline model', 'against'], where=vs_bm['p'].notna())
vs_bm['verdict'] = verdict(vs_bm['dR2 low'], vs_bm['dR2 high'])
vs_bm = sort_bm(vs_bm)
write_csv(vs_bm, os.path.join(EVAL_DIR, 'overall_vs_baseline_models.csv'))
bm_sum = bm_summary(vs_bm, ['combo'])
write_csv(bm_sum, os.path.join(EVAL_DIR, 'overall_vs_baseline_models_summary.csv'))
print(f'{len(vs_bm)} comparisons with the baseline models ->', os.path.join(EVAL_DIR, 'overall_vs_baseline_models.csv'))


def vs_baseline_models_figure(start):
    t = vs_bm[vs_bm['start'] == start]
    bms = [b for b in BASELINE_MODELS if b in set(t['baseline model'])]
    refs = [a for a in BM_AGAINST if a in set(t['against'])]
    if not bms:
        return
    combos = sorted(set(t['combo']), key=combo_order)
    models = sorted(set(t['model']), key=model_order)
    norm = delta_norm(t.loc[~t['model'].isin(NOISE_MODELS), 'dR2'], floor=0.02)
    fig, axes = plt.subplots(len(refs), len(bms), figsize=((0.8 * len(combos) + 0.6) * len(bms) + 2.8,
                                                           (0.36 * len(models) + 1.2) * len(refs) + 1.6),
                             squeeze=False, constrained_layout=True)
    for i, against in enumerate(refs):
        for j, b in enumerate(bms):
            ax = axes[i][j]
            g = t[(t['baseline model'] == b) & (t['against'] == against)]
            m = g.pivot(index='model', columns='combo', values='dR2').reindex(index=models, columns=combos)
            sig = g.assign(s=g['verdict'].isin(['better', 'worse']).astype(float)).pivot(
                index='model', columns='combo', values='s').reindex(index=models, columns=combos).fillna(0) > 0.5
            im = heatmap(ax, m.to_numpy(dtype=float), [model_label(k) for k in models], [short_combo(c) for c in combos],
                         DIVERGING, norm, labels=[[fmt_delta(x) if np.isfinite(x) else '' for x in row]
                                                  for row in m.to_numpy(dtype=float)], bold=sig.to_numpy(dtype=bool))
            ax.set_title(f'minus {bm_label(b, against)}', fontsize=10)
            if j:
                ax.set_yticklabels([])
            if i < len(refs) - 1:
                ax.set_xticklabels([])
    fig.colorbar(im, ax=list(axes.flat), shrink=0.5, extend='both',
                 label='R²(row model) − R²(baseline model); bold: 95 % interval excludes 0')
    fig.suptitle(f'{start.capitalize()}: every model of 11-15 against the baseline models, whole continent',
                 x=0.01, ha='left', fontsize=11)
    save_figure(fig, f'overall_vs_baseline_models_{start}')


if len(vs_bm):
    for start in [s for s in START_ORDER if s in set(vs_bm['start'])]:
        for b in [b for b in BASELINE_MODELS if b in set(vs_bm.loc[vs_bm['start'] == start, 'baseline model'])]:
            for against in BM_AGAINST:
                t = vs_bm[(vs_bm['start'] == start) & (vs_bm['baseline model'] == b) & (vs_bm['against'] == against)]
                if len(t):
                    print(f'\n{start.upper()}: every model minus {bm_label(b, against)}, ΔR² [95 %] (▲ better, ▼ worse)')
                    display(delta_table(t.rename(columns={'verdict': 'vs baseline'}), 'model', 'combo', model_order,
                                        combo_order).rename(index=model_label, columns=short_combo))
    if len(bm_sum):
        m = bm_sum.assign(cell=bm_sum.apply(consensus_cell, axis=1),
                          row=[f'{s} | minus {bm_label(b, a)}' for s, b, a in
                               zip(bm_sum['start'], bm_sum['baseline model'], bm_sum['against'])])
        rows_order = list(dict.fromkeys(m['row']))
        m = m.pivot(index='row', columns='combo', values='cell')
        m = m.reindex(index=rows_order, columns=sorted(m.columns, key=combo_order)).fillna('')
        print('\nover the ten models (without satclip_only and the baseline models): median ΔR² against the baseline '
              'model (models better / models; ▲ significantly better, ▼ worse)')
        display(m.rename(columns=short_combo))
    for start in [s for s in START_ORDER if s in set(vs_bm['start'])]:
        vs_baseline_models_figure(start)
else:
    print('no baseline model (BASELINE_MODELS) among the runs of 11-15 yet: run 11-15 with --array=10-11')

# 7. Appendix: pretrained minus scratch

For every family, model and input with a run from pretrained weights and one from scratch: ΔR² = R²(pretrained) − R²(scratch) on the test clusters both have, with its interval. MOSAIKS and the MLP baseline (no pretrained weights) and `satclip_only` (fixed SatCLIP values) are the same model in both starts, so their difference is training noise.

In [ ]:
cross = []
if CROSS_START:
    rid = {(r.family, r.model, r.combo, r.start): r.run_id for r in USED.itertuples()}
    for (fam, model, combo, start), run_id in rid.items():
        other = rid.get((fam, model, combo, 'scratch'))
        if start != 'pretrained' or other is None:
            continue
        res = compare(Y, COLUMNS[other], COLUMNS[run_id], ALL, BOOT, FOLD_CODE)[CONTINENT]
        cross.append({'start': 'pretrained', 'family': fam, 'model': model, 'combo': combo, 'n': res['n'],
                      'R2 pretrained': res.get('R2 b', np.nan), 'R2 scratch': res.get('R2 a', np.nan),
                      **{k: res.get(k, np.nan) for k in PAIR_COLS[3:]}})
cross = pd.DataFrame(cross, columns=['start', 'family', 'model', 'combo', 'n', 'R2 pretrained', 'R2 scratch'] +
                     PAIR_COLS[3:])
if len(cross):
    cross = add_q(cross, ['family'], where=cross['p'].notna(), split=baseline_split(cross))
    cross['verdict'] = verdict(cross['dR2 low'], cross['dR2 high'])
    cross = sort_runs(cross).drop(columns='start').reset_index(drop=True)
    write_csv(cross, os.path.join(EVAL_DIR, 'overall_pretrained_minus_scratch.csv'))
    t = cross[cross['family'] == 'ten_models']
    if len(t):
        m = t.pivot(index='model', columns='combo', values='dR2')
        m = m.reindex(index=sorted(m.index, key=model_order), columns=sorted(m.columns, key=combo_order))
        sig = t.assign(s=t['verdict'].isin(['better', 'worse']).astype(float)).pivot(index='model', columns='combo',
                                                                                      values='s')
        sig = (sig.reindex(index=m.index, columns=m.columns).fillna(0) > 0.5).to_numpy(dtype=bool)
        fig, ax = plt.subplots(figsize=(0.8 * m.shape[1] + 2.8, 0.36 * m.shape[0] + 2.2), constrained_layout=True)
        im = heatmap(ax, m.to_numpy(dtype=float), [model_label(k) for k in m.index], [short_combo(c) for c in m.columns],
                     DIVERGING, delta_norm(t['dR2'], floor=0.02),
                     labels=[[fmt_delta(x) if np.isfinite(x) else '' for x in row] for row in m.to_numpy(dtype=float)],
                     bold=sig)
        fig.colorbar(im, ax=ax, shrink=0.8, extend='both', label='R²(pretrained) − R²(scratch)')
        ax.set_title('The models of 11-15: pretrained minus scratch (bold: 95 % interval excludes 0)', fontsize=10)
        save_figure(fig, 'overall_pretrained_minus_scratch')
    display(cross.assign(model=cross['model'].map(model_label), combo=cross['combo'].map(short_combo)).round(4))
else:
    print('no model and input with both a pretrained and a scratch run' if CROSS_START else 'CROSS_START = False')